<a href="https://colab.research.google.com/github/hamnasz/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran/blob/main/SI26_Week1_humna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Research Task (Step 5)

**What is OCR (Optical Character Recognition)?**

OCR is a technology that takes an image containing text — a scanned page, a photo, a screenshot — and converts it into actual machine-readable text that a computer can search, edit, or read aloud. Instead of a document just being a picture of letters, the software identifies each character and outputs it as real text data instead of pixels. It's the same idea behind Google Lens reading a street sign, or a scanner app turning a photographed page into an editable Word document.

**Why is Urdu OCR harder than English OCR?**

Urdu is written in the Nastaliq script, which is cursive and diagonal by default — letters change shape depending on whether they sit at the start, middle, or end of a word, and they slope and overlap in a way Latin letters don't. English has 26 fairly fixed letter shapes that look almost the same everywhere; a single Urdu character can have four or more different forms depending on its position in a word, so a model has to learn far more visual variation just to recognise the "same" letter. On top of that, Urdu is written right-to-left, and there simply aren't as many large labeled Urdu datasets available compared to English, so models have less data to learn from in the first place.

**What are 2 real-world situations where Urdu OCR would be useful?**

One is digitizing old Urdu newspapers, books, and government archives so they can be searched and preserved, instead of sitting as unindexed scanned images or physical copies that degrade over time. Another is automating data entry from Urdu paperwork — CNIC application forms, utility bills, handwritten prescriptions in Pakistani hospitals — so staff don't have to manually retype everything from a photo or scan.

*(Also copied into the GitHub README as required by the handout.)*


---

**Update, after the Week 4 training audit:** the `books`/`newspaper` sections further
down used to copy whole scanned pages into `labels.csv` as if each page were a single
training line, labeled with either a generic section-type placeholder (`books`) or an
unproofread best-effort headline (`newspaper`) — neither describes what's actually
printed across the full page. 74 of 353 rows in the live dataset turned out to be built
this way. Fixed below: pages now get segmented into individual line crops first, and
only real, verified per-line transcriptions go into `labels.csv`. See the note right
before Step 22b for the details.

In [ ]:
import csv
import os
import shutil
import subprocess
import sys
import zipfile


In [ ]:
GDRIVE_FILE_ID = "1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T"


In [ ]:
# Junk/intermediate files (raw downloads, zip extracts) live under
# Other so they never mix with the real data/ deliverable.
OTHER_DIR = "Other"
DOWNLOAD_DIR = os.path.join(OTHER_DIR, "utrset_real_download")
ZIP_PATH = os.path.join(DOWNLOAD_DIR, "utrset_real.zip")
EXTRACT_DIR = os.path.join(DOWNLOAD_DIR, "extracted")


In [ ]:
OUT_DIR = "data/raw/other"
LABELS_CSV = "data/labels.csv"
N_SAMPLES = 60


In [ ]:
def merge_labels(new_rows, labels_csv=LABELS_CSV):
    """Merge new_rows into labels_csv, keyed by image path.

    Safe to call multiple times (or run this notebook top-to-bottom more
    than once) with the same rows -- it will NOT create duplicate lines.
    If an image currently has a FILL_IN_URDU_TEXT_HERE placeholder and
    new_rows now has a real label for that same path, the real label
    overwrites the placeholder. A real label already on file is never
    overwritten by a placeholder.
    """
    existing_rows = []
    if os.path.exists(labels_csv):
        with open(labels_csv, "r", encoding="utf-8") as f:
            existing_rows = list(csv.DictReader(f))

    rows_by_path = {row["image"]: row for row in existing_rows}
    for row in new_rows:
        path = row["image"]
        already_real = (
            path in rows_by_path
            and rows_by_path[path]["text"] != "FILL_IN_URDU_TEXT_HERE"
        )
        if already_real and row["text"] == "FILL_IN_URDU_TEXT_HERE":
            continue  # never clobber a real label with a placeholder
        rows_by_path[path] = row

    all_rows = list(rows_by_path.values())
    os.makedirs(os.path.dirname(labels_csv) or ".", exist_ok=True)
    with open(labels_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image", "text"])
        writer.writeheader()
        writer.writerows(all_rows)

    return all_rows


In [ ]:
def ensure_gdown():
    try:
        import gdown  # noqa: F401
        return
    except ImportError:
        print("Installing gdown...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown"])


def download():
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    if os.path.exists(ZIP_PATH):
        print(f"Already downloaded: {ZIP_PATH}")
        return

    ensure_gdown()
    print("Downloading UTRSet-Real from Google Drive (this is ~hundreds of MB, may take a few minutes)...")
    import gdown
    gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    if not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) == 0:
        raise RuntimeError(
            "Download failed or produced an empty file. Google Drive "
            "sometimes blocks automated downloads of large files with a "
            "'too many downloads' warning page instead of the real file. "
            "If this happens: open the link in your own browser "
            "(https://drive.google.com/file/d/1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T/view), "
            "click through the warning, download manually, then upload "
            "the zip to Colab and re-run this script -- it'll detect the "
            "existing zip and skip straight to extraction."
        )
    print(f"Downloaded: {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)")


In [ ]:
def extract():
    if os.path.exists(EXTRACT_DIR) and os.listdir(EXTRACT_DIR):
        print(f"Already extracted: {EXTRACT_DIR}")
        return
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extracted.")


In [ ]:
def find_gt_file(root):
    """Locate the ground-truth label file -- commonly gt.txt per the
    dataset author's own repo convention, but search broadly in case
    the zip structure differs."""
    candidates = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower() in ("gt.txt", "ground_truth.txt", "labels.txt"):
                candidates.append(os.path.join(dirpath, fname))
    return candidates


In [ ]:
def find_images(root):
    images = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                images.append(os.path.join(dirpath, fname))
    return images


In [ ]:
def main():
    download()
    extract()

    gt_files = find_gt_file(EXTRACT_DIR)
    images = find_images(EXTRACT_DIR)

    print(f"\nFound {len(gt_files)} ground-truth file(s), {len(images)} image(s) in the extracted archive.")

    os.makedirs(OUT_DIR, exist_ok=True)

    new_rows = []

    if gt_files:
        gt_path = gt_files[0]
        print(f"Parsing ground truth from: {gt_path}")
        with open(gt_path, "r", encoding="utf-8") as f:
            lines = [l.strip() for l in f if l.strip()]

        gt_dir = os.path.dirname(gt_path)
        count = 0
        for line in lines:
            if count >= N_SAMPLES:
                break
            if "\t" in line:
                img_rel_path, text = line.split("\t", 1)
            elif " " in line:
                img_rel_path, text = line.split(" ", 1)
            else:
                continue

            src_img_path = os.path.join(gt_dir, img_rel_path)
            if not os.path.exists(src_img_path):
                src_img_path = os.path.join(EXTRACT_DIR, img_rel_path)
            if not os.path.exists(src_img_path):
                continue

            fname = f"utrset_{count:03d}.png"
            dst_path = os.path.join(OUT_DIR, fname)
            shutil.copy(src_img_path, dst_path)
            new_rows.append({"image": dst_path, "text": text})
            count += 1

        print(f"Copied {count} images with verified ground-truth labels.")

    elif images:
        print("WARNING: No ground-truth file found automatically.")
        print("Copying a sample of images anyway -- you'll need to find")
        print("the label source manually (check the extracted folder")
        print(f"structure at {EXTRACT_DIR}) or label these by eye.")
        for i, img_path in enumerate(images[:N_SAMPLES]):
            fname = f"utrset_{i:03d}.png"
            dst_path = os.path.join(OUT_DIR, fname)
            shutil.copy(img_path, dst_path)
            new_rows.append({"image": dst_path, "text": "FILL_IN_URDU_TEXT_HERE"})
    else:
        raise RuntimeError(
            f"No images or ground-truth files found under {EXTRACT_DIR}. "
            f"Check the actual folder structure with: "
            f"!find {EXTRACT_DIR} -maxdepth 3"
        )

    all_rows = merge_labels(new_rows)

    print(f"labels.csv now has {len(all_rows)} total entries.")
    print(f"\nDataset license: CC BY-NC-SA 4.0 (non-commercial, research use only).")
    print("Cite in your README:")
    print("""
  Rahman, A., Ghosh, A., Arora, C. (2023). UTRNet: High-Resolution Urdu
  Text Recognition in Printed Documents. In: Document Analysis and
  Recognition - ICDAR 2023. Springer Nature Switzerland.
""")


if __name__ == "__main__":
    main()


In [ ]:
!pip install Pillow ipywidgets -q
print("Dependencies installed.")


In [ ]:
import os

folders = [
    'data/raw/newspaper',
    'data/raw/books',
    'data/raw/signboards',
    'data/raw/synthetic',
    'data/raw/other',
]


In [ ]:
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'Created: {folder}')

print('Folder structure ready!')


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import os
import urllib.request
import csv


In [ ]:
# 20 short Urdu sentences for synthetic image generation (was 10 -- handout
# asks for 20+ from this source).
urdu_texts = [
    "پاکستان زندہ باد",
    "آج کا موسم خوشگوار ہے",
    "تعلیم ہر انسان کا حق ہے",
    "کراچی پاکستان کا سب سے بڑا شہر ہے",
    "محنت کامیابی کی کنجی ہے",
    "علم انسان کو روشنی دیتا ہے",
    "وقت کی قدر کرنی چاہیے",
    "صحت سب سے بڑی نعمت ہے",
    "دوستی ایک قیمتی رشتہ ہے",
    "محنت کبھی رائیگاں نہیں جاتی",
    "پانی زندگی کی علامت ہے",
    "کتاب انسان کی بہترین دوست ہے",
    "سچ بولنا ایک اچھی عادت ہے",
    "لاہور پاکستان کا دل ہے",
    "بچے ملک کا مستقبل ہیں",
    "صبر کا پھل میٹھا ہوتا ہے",
    "ورزش صحت کے لیے ضروری ہے",
    "اردو ایک خوبصورت زبان ہے",
    "درخت ہمیں آکسیجن دیتے ہیں",
    "نظم و ضبط کامیابی کی کنجی ہے",
]


In [ ]:
os.makedirs('data/raw/synthetic', exist_ok=True)


In [ ]:
FONT_PATH = "NotoNastaliqUrdu-Regular.ttf"
if not os.path.exists(FONT_PATH):
    font_url = (
        "https://raw.githubusercontent.com/google/fonts/main/ofl/"
        "notonastaliqurdu/NotoNastaliqUrdu%5Bwght%5D.ttf"
    )
    try:
        urllib.request.urlretrieve(font_url, FONT_PATH)
        print(f"Downloaded font to {FONT_PATH}")
    except Exception as e:
        print(f"Could not auto-download font ({e}).")
        print("Manually download 'Noto Nastaliq Urdu' from fonts.google.com,")
        print(f"upload it to Colab's file panel, named {FONT_PATH}")


In [ ]:
font = ImageFont.truetype(FONT_PATH, 36)


In [ ]:
synthetic_labels = []
dummy_img = Image.new('RGB', (10, 10))
dummy_draw = ImageDraw.Draw(dummy_img)

for i, text in enumerate(urdu_texts):
    # direction='rtl' + language='ur' hands shaping to Pillow's raqm/HarfBuzz
    # backend, which reads the original letters and picks the correct
    # joined/contextual glyph forms itself -- no manual reshaping needed,
    # and it's what actually matches how Noto Nastaliq Urdu is built to be used.
    bbox = dummy_draw.textbbox((0, 0), text, font=font, direction='rtl', language='ur')
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    img = Image.new('RGB', (text_w + 40, text_h + 40), color='white')
    draw = ImageDraw.Draw(img)
    draw.text((20 - bbox[0], 20 - bbox[1]), text, fill='black', font=font,
              direction='rtl', language='ur')

    save_path = f'data/raw/synthetic/urdu_{i+1}.png'
    img.save(save_path)
    synthetic_labels.append({'image': save_path, 'text': text})
    print(f'Generated: {save_path} -- {text}')


In [ ]:
print(f'Done! {len(synthetic_labels)} synthetic images in data/raw/synthetic/')


In [ ]:
all_rows = merge_labels(synthetic_labels)
print(f'labels.csv now has {len(all_rows)} total entries.')


## Step 22a: Replace signboards with the updated real photo set

The `signboards` folder previously had 99 substitute images pulled from another
dataset. Upload `Signboard.zip` (100 real Urdu signboard photos) into
`S126-Week1/Other/` in Colab's file panel, then run the cell below **before**
re-running Step 22 -- it wipes the old substitute images and their
`labels.csv` rows so nothing stale or duplicated lingers.


In [ ]:
import os, shutil, csv

# 1. Wipe the old signboards folder (the UTRSet substitute images)
old_dir = "data/raw/signboards"
if os.path.exists(old_dir):
    shutil.rmtree(old_dir)
os.makedirs(old_dir, exist_ok=True)

# 2. Remove old signboards rows from labels.csv so nothing stale lingers
LABELS_CSV = "data/labels.csv"
if os.path.exists(LABELS_CSV):
    with open(LABELS_CSV, "r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    rows = [r for r in rows if not r["image"].startswith(old_dir)]
    with open(LABELS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image", "text"])
        writer.writeheader()
        writer.writerows(rows)

print("Old signboards cleared. Now re-run Step 22 (the category_to_folder cell) "
      "to bring in the new Signboard.zip.")


In [ ]:
import os
import shutil
import zipfile

# Same convention as the UTRSet download above: raw zips you upload and
# their extracted contents are junk once the images are copied into data/,
# so they live under Other, not the repo root.
OTHER_DIR = "Other"
os.makedirs(OTHER_DIR, exist_ok=True)

# category -> correct target folder name (must match `folders` above).
# This was the bug: out_dir used to be f"data/raw/{category.lower()}",
# which produced "book" and "signboard" (singular) instead of the
# "books" / "signboards" (plural) folders everything else expects.
#
# Book/Newspaper now land in *_pages holding folders instead of the tracked
# books/newspaper folders directly -- these zips contain full scanned pages,
# not single lines, and copying a whole page straight into labels.csv as "one
# training example" is exactly the bug Step 22b below fixes. Signboard is
# unchanged: those are individual sign photos, not multi-line pages.
category_to_folder = {
    "Book": "books_pages",
    "Newspaper": "newspaper_pages",
    "Signboard": "signboards",
}

for category, folder_name in category_to_folder.items():
    # Upload Book.zip / Newspaper.zip / Signboard.zip into Other
    zip_path = os.path.join(OTHER_DIR, f"{category}.zip")
    extract_to = os.path.join(OTHER_DIR, f"{category.lower()}_extract")
    out_dir = f"data/raw/{folder_name}"  # *_pages for Book/Newspaper, straight to the tracked folder for Signboard

    # Check if the zip file exists to avoid crashing
    if not os.path.exists(zip_path):
        print(f"Warning: '{zip_path}' not found. Skipping...")
        continue

    # Create output directory
    os.makedirs(out_dir, exist_ok=True)

    # Extract the zip
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)

    # Find all images regardless of nested folder
    images = []
    for root, _, files in os.walk(extract_to):
        for f in sorted(files):
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                images.append(os.path.join(root, f))

    # Copy and rename images sequentially
    for i, src in enumerate(sorted(images), start=1):
        # Keep the original file extension to prevent file corruption
        _, ext = os.path.splitext(src)

        # Format the new name (e.g., book_001.jpg)
        new_name = f"{category.lower()}_{i:03d}{ext.lower()}"
        dst = os.path.join(out_dir, new_name)

        shutil.copy(src, dst)

    print(f"Copied and renamed {len(images)} images from '{zip_path}' into '{out_dir}'")


## Step 22b: Segment Pages Into Real Training Lines

*(Not in the handout -- added after the Week 4 training audit found this was broken.)*

The cell above now drops Book.zip/Newspaper.zip pages into `books_pages`/
`newspaper_pages` instead of the tracked `books`/`newspaper` folders. Those are full
scanned pages -- a heading, maybe a photo, several paragraphs -- and a `labels.csv` row
needs its `text` to match what's actually in its `image`, not a page's section title.
The old approach (`NEWSPAPER_TRANSCRIPTIONS`, a hand-typed dict of one headline per page,
explicitly flagged in its own comment as an unproofread "strong first draft") got used
as real ground truth anyway. `books` was worse — every one of its 35 labels turned out
to be a generic book-section name (Preface, Chapter 1, Table of Contents...) assigned in
filename order, unrelated to any specific page's content.

This cell segments each page into individual line crops using a horizontal projection
profile (count dark pixels per row; gaps between lines show up as near-zero bands).
Tested against real pages from this dataset: 24/24 clean bands on one, but only 6/10
clean on another where a photo header and tightly-set Nastaliq lines threw it off — so it
flags anything whose height looks off (too tall = probably a merged pair of lines or a
non-text region like a photo; too short = probably noise) instead of silently trusting
every band. Unflagged crops are reliable enough to transcribe directly; flagged ones are
worth a glance before deciding whether to keep them at all.

**New in this pass:** if `labels.csv` already has a full-page transcription for a page
(this project's `books`/`newspaper` rows do, from an earlier AI-transcription pass), that
text gets captured *before* this section touches labels.csv, and used to generate a
**suggested** transcription for each line crop — not by trusting an OCR engine's raw
output, but by roughly reading each crop (Tesseract's Urdu model, upscaled 3x — still
weak on Nastaliq, tested at ~30-45% string-similarity to the true line) and fuzzy-matching
that rough read against the *known-correct* full-page text to find which part of it this
crop most likely corresponds to. That confidence score travels with the suggestion into
the labeling widget below, pre-filling the text box instead of leaving it blank — turning
"transcribe from scratch" into "review and correct," which is a lot faster, especially at
the scale of an entire book/newspaper folder. Nothing gets written to `labels.csv` from
this automatically -- every row still requires a human click in the widget before it
counts as labeled.

In [ ]:
import csv

# Capture whatever full-page text labels.csv already has for books/newspaper BEFORE
# the cleanup cell below removes those rows -- this is the reading context the
# suggestion generator uses. Keyed by basename so it still matches after re-extraction
# renames the containing folder (data/raw/books/book_002.png -> .../books_pages/book_002.png).
page_text_context = {}
if os.path.exists(LABELS_CSV):
    with open(LABELS_CSV, "r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            img = row["image"]
            if img.startswith("data/raw/books/") or img.startswith("data/raw/newspaper/"):
                page_text_context[os.path.basename(img)] = row["text"]
print(f"Captured existing full-page text for {len(page_text_context)} page(s) "
      f"to use as reading context for the line-level suggestions below.")

In [ ]:
import numpy as np
import statistics
from PIL import Image


def segment_page_into_lines(path, out_dir, pad=4):
    """Crop one scanned page into individual line images via horizontal projection
    profiling. Returns (n_clean, n_flagged). Flagged crops (unusual height relative to
    the page's own median line height) get a FLAGGED_ prefix -- eyeball those before
    trusting them; they're often a photo/decorative region or two merged lines rather
    than one clean line of text.
    """
    im = Image.open(path).convert("RGB")
    arr = np.array(im.convert("L"))
    thresh = arr.mean() - 0.3 * arr.std()
    binary = (arr < thresh).astype(np.uint8)
    row_ink = binary.sum(axis=1)
    if row_ink.max() == 0:
        return 0, 0  # blank page, nothing to segment

    is_gap = row_ink <= row_ink.max() * 0.03
    bands, in_band = [], False
    for i, g in enumerate(is_gap):
        if not g and not in_band:
            start, in_band = i, True
        elif g and in_band:
            bands.append((start, i))
            in_band = False
    if in_band:
        bands.append((start, len(is_gap)))

    merged = []
    for b in bands:
        if merged and b[0] - merged[-1][1] < 4:
            merged[-1] = (merged[-1][0], b[1])
        else:
            merged.append(list(b))
    if not merged:
        return 0, 0

    heights = [e - s for s, e in merged]
    med = statistics.median(heights)
    os.makedirs(out_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(path))[0]
    n_clean = n_flagged = 0
    for i, (s, e) in enumerate(merged):
        flagged = not (0.5 * med <= (e - s) <= 1.7 * med)
        crop = im.crop((0, max(0, s - pad), im.width, min(im.height, e + pad)))
        tag = "FLAGGED_" if flagged else ""
        crop.save(os.path.join(out_dir, f"{base}_{tag}line{i:02d}.png"))
        n_clean += not flagged
        n_flagged += flagged
    return n_clean, n_flagged


page_to_line_dirs = {
    "data/raw/books_pages": "data/raw/books",
    "data/raw/newspaper_pages": "data/raw/newspaper",
}

for pages_dir, lines_dir in page_to_line_dirs.items():
    if not os.path.isdir(pages_dir):
        print(f"{pages_dir}: not found, skipping (upload/extract the zip above first)")
        continue
    total_clean = total_flagged = total_pages = 0
    for fname in sorted(os.listdir(pages_dir)):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        nc, nf = segment_page_into_lines(os.path.join(pages_dir, fname), lines_dir)
        total_clean += nc
        total_flagged += nf
        total_pages += 1
    print(f"{pages_dir} -> {lines_dir}: {total_pages} pages segmented, "
          f"{total_clean} clean line crops, {total_flagged} flagged (need a manual look)")

In [ ]:
import difflib
import json as _json

try:
    import pytesseract
except ImportError:
    get_ipython().system("apt-get install -y tesseract-ocr-urd -q")
    get_ipython().system("pip install pytesseract -q")
    import pytesseract


def suggest_line_text(crop_path, full_text, crop_index, n_crops, ocr_lang="urd"):
    """Rough-OCR one crop (Tesseract, upscaled 3x -- still weak on Nastaliq, but not
    random), then fuzzy-match that rough read against a window of full_text centered on
    this crop's PROPORTIONAL expected position (crop_index / n_crops). Independent
    per-crop windows -- one bad OCR read can't derail the crops after it, unlike a
    moving cursor would. Returns (suggested_text, confidence_ratio).
    """
    img = Image.open(crop_path)
    img_up = img.resize((img.width * 3, img.height * 3), Image.LANCZOS)
    rough = pytesseract.image_to_string(img_up, lang=ocr_lang, config="--psm 7").strip().replace("\n", " ")

    text_len = len(full_text)
    if not rough or text_len == 0 or n_crops == 0:
        return "", 0.0

    expected_center = int((crop_index + 0.5) / n_crops * text_len)
    half_window = max(int(1.5 * text_len / n_crops), 80)
    w_start = max(0, expected_center - half_window)
    w_end = min(text_len, expected_center + half_window)
    window = full_text[w_start:w_end]
    if not window:
        return "", 0.0

    rl = max(len(rough), 8)
    best_ratio, best_span = 0.0, ""
    step = max(1, rl // 4)
    for start in range(0, max(1, len(window) - rl // 2), step):
        candidate = window[start:start + rl + 15]
        if not candidate:
            continue
        ratio = difflib.SequenceMatcher(None, rough, candidate, autojunk=False).ratio()
        if ratio > best_ratio:
            best_ratio, best_span = ratio, candidate
    return best_span.strip(), best_ratio


SUGGESTIONS_PATH = "data/line_suggestions.json"
suggestions = {}  # image path (as it will appear in labels.csv) -> {"text":..., "confidence":...}

for pages_dir, lines_dir in page_to_line_dirs.items():
    if not os.path.isdir(lines_dir):
        continue
    # group crops by source page (everything before "_line" or "_FLAGGED_line" in the filename)
    by_page = {}
    for fname in sorted(os.listdir(lines_dir)):
        if "_line" not in fname:
            continue
        page_key = fname.split("_line")[0].replace("_FLAGGED", "") + os.path.splitext(fname)[1]
        by_page.setdefault(page_key, []).append(fname)

    for page_key, crop_fnames in by_page.items():
        full_text = page_text_context.get(page_key)
        if not full_text:
            continue  # no captured context for this page -- widget falls back to blank
        crop_fnames = sorted(crop_fnames)
        n = len(crop_fnames)
        for idx, fname in enumerate(crop_fnames):
            crop_path = f"{lines_dir}/{fname}"
            text, ratio = suggest_line_text(crop_path, full_text, idx, n)
            suggestions[crop_path] = {"text": text, "confidence": round(ratio, 3)}

with open(SUGGESTIONS_PATH, "w", encoding="utf-8") as f:
    _json.dump(suggestions, f, ensure_ascii=False, indent=1)

n_with_text = sum(1 for v in suggestions.values() if v["text"])
avg_conf = (sum(v["confidence"] for v in suggestions.values()) / len(suggestions)) if suggestions else 0
print(f"Generated {n_with_text}/{len(suggestions)} suggestions -> {SUGGESTIONS_PATH} "
      f"(mean confidence {avg_conf:.2f})")
print("These are rough starting points, not verified text -- the widget below shows the "
      "confidence score next to each one so you know how much to double-check.")

# Sanity check: captured page_text_context assumes re-extracting Book.zip/Newspaper.zip
# reproduces the same book_NNN/newspaper_NNN numbering as the original extraction that
# produced the text currently in labels.csv (true as long as the zip's internal file
# order hasn't changed, which it normally hasn't). Flag it plainly if that assumption
# didn't hold, rather than silently leaving some pages with no suggestions.
extracted_pages = set()
for _pages_dir in page_to_line_dirs:
    if os.path.isdir(_pages_dir):
        extracted_pages.update(os.listdir(_pages_dir))
orphaned = set(page_text_context) - extracted_pages
if orphaned:
    print(f"\nWarning: {len(orphaned)} page(s) had captured text but no matching extracted "
          f"page filename (re-extraction may have renumbered things differently than "
          f"before) -- no suggestions were generated for these: {sorted(orphaned)[:5]}"
          f"{' ...' if len(orphaned) > 5 else ''}. Their line crops will still show up in "
          "the widget below with a blank suggestion instead of a pre-filled one.")

In [ ]:
import csv

# One-time cleanup: if this notebook ran before today's fix, labels.csv still has the
# old full-page rows (data/raw/books/book_NNN.png, data/raw/newspaper/newspaper_NNN.png)
# labeled with the broken section-name/unproofread-headline text. Those exact paths are
# never written again now that Book/Newspaper extraction targets *_pages -- remove them
# so the new, real line crops (same folder, different filenames -- see the *_lineNN.png
# naming above) aren't sitting alongside stale, wrong-granularity rows.
if os.path.exists(LABELS_CSV):
    with open(LABELS_CSV, "r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    def is_old_flat_page_row(row):
        img = row["image"]
        return (
            (img.startswith("data/raw/books/") or img.startswith("data/raw/newspaper/"))
            and "_line" not in img  # new crops always have _lineNN in the name
        )

    kept = [r for r in rows if not is_old_flat_page_row(r)]
    removed = len(rows) - len(kept)
    if removed:
        with open(LABELS_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["image", "text"])
            writer.writeheader()
            writer.writerows(kept)
    print(f"Removed {removed} old full-page row(s) from labels.csv "
          f"({len(rows)} -> {len(kept)}). The page images themselves are untouched on "
          "disk under books_pages/newspaper_pages if you want to re-segment later.")
else:
    print("labels.csv doesn't exist yet -- nothing to clean up.")

### Reviewing the suggested transcriptions

`data/raw/books/` and `data/raw/newspaper/` now hold individual line crops
(`..._line00.png`, `..._line01.png`, ...) instead of full pages, and
`data/line_suggestions.json` has a rough starting-point transcription for most of
them (see Step 22b above for how -- short version: a weak OCR read of the crop,
fuzzy-matched against the full-page text this project already had, to find which part
of that known-correct text this specific line most likely is).

**Treat every suggestion as a draft, not an answer.** Confidence scores in testing
ranged from ~0.3 to ~0.6 (string similarity between the rough OCR and the matched
text) -- useful enough to make reviewing much faster than transcribing from scratch,
not reliable enough to accept blindly. The widget below shows each suggestion
pre-filled in the text box, with its confidence score in the progress line: read the
crop, compare it to the pre-filled text, fix whatever's wrong (or clear it and retype
if the suggestion is unrelated to the crop -- this happens, especially on `FLAGGED_`
crops or ones near an image/photo region), then save and move on. Skip anything
where the crop itself isn't real Urdu text (a photo fragment, a decorative border)
rather than forcing a transcription onto it.

If you'd rather use an external AI vision tool for a batch instead of typing directly
in the widget, the same prompt from before still works -- paste a few crops with:

> These are cropped lines from scanned Urdu book/magazine pages (Nastaliq typeface).
> For each image, transcribe *only* the Urdu text visible in that specific crop --
> exactly what's printed, nothing added or inferred, nothing summarized. Preserve
> numerals, punctuation, and diacritics exactly as shown. If a word is genuinely
> illegible, write `[unclear]` at that position instead of guessing. If a crop has no
> readable Urdu text at all, respond `NO_TEXT` for that image. Output as a numbered
> list matching the image order, one line of transcription per image, nothing else.

...then paste the results into the widget's text box in place of the pre-filled
suggestion before saving.

## Step 25a: Transcriptions for the real Signboard.zip photos

Best-effort transcriptions for the 30 real Urdu signboard photos in
`Signboard.zip`, read directly off the images -- same convention as the
newspaper transcriptions above: short label per image (shop/place name or
clearest headline), not a full paragraph.

**IMPORTANT: proofread these against the actual images before committing,
especially the ones flagged below.** Street photos have small, angled,
sun-glared, or partially-obstructed text, so this is a strong first draft,
not guaranteed-correct ground truth:
- `signboard_003.jpg` (rice bag close-up) -- Urdu paragraph text too
  blurred/cropped to transcribe reliably, left as a placeholder.
- `signboard_015.jpg` -- this frame (Vital Tea ad) has no visible Urdu
  text at all, left as a placeholder -- consider swapping this photo out.
- `signboard_009.jpg`, `signboard_013.jpg`, `signboard_020.jpg`,
  `signboard_022.jpg` -- part of the sign is cut off / glare-obscured, so
  the first word or two is a best guess.
- `signboard_021.jpg` -- only "بسم اللہ" is visible; the rest of the
  board is out of frame.

In [ ]:
# Filenames below match what Step 22's category_to_folder cell produces:
# sorted(images) from Signboard.zip, renamed signboard_001.jpg, signboard_002.jpg, ...
SIGNBOARD_TRANSCRIPTIONS = {
    "signboard_001.jpg": "جامع مسجد عائشہ صدیقہؓ",
    "signboard_002.jpg": "بلال جنرل سٹور",
    "signboard_003.jpg": "FILL_IN_URDU_TEXT_HERE",  # rice bag, text too blurred to read reliably
    "signboard_004.jpg": "کراچی چاٹ نصیب",
    "signboard_005.jpg": "فاسٹ کیش",
    "signboard_006.jpg": "شعبان بکس اینڈ سٹیشنرز",
    "signboard_007.jpg": "کرنٹ",
    "signboard_008.jpg": "خان من بیف اینڈ چکن شاپ",
    "signboard_009.jpg": "فوٹو اسٹیٹ اینڈ کلر پرنٹ",  # first word partly obscured, double-check
    "signboard_010.jpg": "آپ کے پیمنٹ کارڈز کا تحفظ",
    "signboard_011.jpg": "بسم اللہ ملک شیک",
    "signboard_012.jpg": "شکایات کے لیے رابطہ کریں",
    "signboard_013.jpg": "فروٹ اینڈ کولڈ ڈرنکس",  # left edge cut off, double-check
    "signboard_014.jpg": "فری آئل چینج",
    "signboard_015.jpg": "FILL_IN_URDU_TEXT_HERE",  # no Urdu text visible in this frame
    "signboard_016.jpg": "ملکہ فائبر گلاس اینڈ سٹیل ورکس",
    "signboard_017.jpg": "ایک بار ضرور آزمائیے",
    "signboard_018.jpg": "نیو پاکستان ماربل فیکٹری",
    "signboard_019.jpg": "بصورت دیگر ہوٹل انتظامیہ ذمہ دار نہ ہوگی",
    "signboard_020.jpg": "بسم اللہ برگر پوائنٹ",  # last word stylised, double-check
    "signboard_021.jpg": "بسم اللہ",  # rest of board out of frame
    "signboard_022.jpg": "کولڈ ڈرنکس اینڈ ملک شیک",  # left edge cut off, double-check
    "signboard_023.jpg": "ڈینگی سے بچاؤ کی احتیاطی تدابیر",
    "signboard_024.jpg": "نان شاپ",
    "signboard_025.jpg": "کیا آپ کا گھر ڈینگی سے محفوظ ہے؟",
    "signboard_026.jpg": "مکان کرائے کیلئے خالی ہے",
    "signboard_027.jpg": "مغل گارڈن واہ",
    "signboard_028.jpg": "یہاں پر کوڑا کرکٹ پھینکنا منع ہے",
    "signboard_029.jpg": "کینٹ ڈیری",
    "signboard_030.jpg": "خوش آمدید",
}

In [ ]:
manual_folders = ['newspaper', 'books', 'signboards', 'other']  # line crops now live directly under books/ and newspaper/ -- see Step 22b

new_rows = []
for folder in manual_folders:
    folder_path = f'data/raw/{folder}'
    if not os.path.exists(folder_path):
        continue
    for fname in sorted(os.listdir(folder_path)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        img_path = f'{folder_path}/{fname}'
        if folder == 'signboards' and fname in SIGNBOARD_TRANSCRIPTIONS:
            text = SIGNBOARD_TRANSCRIPTIONS[fname]
        else:
            text = 'FILL_IN_URDU_TEXT_HERE'
        new_rows.append({'image': img_path, 'text': text})

all_rows = merge_labels(new_rows)

n_placeholder = sum(1 for r in all_rows if r['text'] == 'FILL_IN_URDU_TEXT_HERE')
print(f'labels.csv now has {len(all_rows)} total entries.')
print(f'  - still placeholders (need manual labeling): {n_placeholder}')
if n_placeholder:
    print()
    print('  Placeholder rows are your books/ and signboards/ images --')
    print('  those still need real Urdu text typed in by hand before you commit.')


In [ ]:
import csv
import os

print("=" * 50)
print("FOLDER STRUCTURE & IMAGE COUNTS")
print("=" * 50)
total_images = 0
for folder in ['newspaper', 'newspaper_pages', 'books', 'books_pages', 'signboards', 'synthetic', 'other']:
    folder_path = f'data/raw/{folder}'
    if os.path.exists(folder_path):
        count = len([f for f in os.listdir(folder_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        total_images += count
        print(f'  {folder:12s}: {count} images')
    else:
        print(f'  {folder:12s}: folder missing!')

print(f'\nTOTAL IMAGES: {total_images}  (target: 100+)')

print("\n" + "=" * 50)
print("LABELS.CSV STATUS")
print("=" * 50)
n_placeholder = 0
if os.path.exists('data/labels.csv'):
    with open('data/labels.csv', 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    n_filled = sum(1 for r in rows if r['text'] != 'FILL_IN_URDU_TEXT_HERE')
    n_placeholder = len(rows) - n_filled
    print(f'  Total rows: {len(rows)}')
    print(f'  Labeled: {n_filled}')
    print(f'  Still need manual labeling: {n_placeholder}')
    print(f'\n  First 3 rows:')
    for r in rows[:3]:
        print(f'    {r["image"]} -> {r["text"][:40]}')
else:
    print('  labels.csv not found!')

if total_images >= 100 and n_placeholder == 0:
    print('\n✅ Ready to commit to GitHub.')
elif total_images >= 100:
    print(f'\n⚠️  Image count OK, but {n_placeholder} labels still need filling in.')
else:
    print(f'\n⚠️  Need {100 - total_images} more images.')


## Manual labeling widget

Fast click-through tool for filling in the remaining `FILL_IN_URDU_TEXT_HERE`
rows. Shows one image at a time with an RTL Urdu text box; each save writes
straight to `labels.csv` so progress is never lost. The queue is sorted with
**signboards first** (the newly-replaced real photos), then books, so you can
knock out the new signboard set in one focused pass before switching folders.

Requires `ipywidgets` (`pip install ipywidgets -q` if not already available
in your Colab environment).


In [ ]:
import csv, os, json as _json
import ipywidgets as widgets
from IPython.display import display, clear_output

LABELS_CSV = "data/labels.csv"

def load_rows():
    with open(LABELS_CSV, "r", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def save_one_label(image_path, text):
    """Update a single row in labels.csv in place, no duplicates."""
    rows = load_rows()
    for r in rows:
        if r["image"] == image_path:
            r["text"] = text
            break
    with open(LABELS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image", "text"])
        writer.writeheader()
        writer.writerows(rows)

# --- build the queue of unlabeled rows, signboards first, then books, then everything else ---
def folder_priority(path):
    # books/newspaper now hold real line crops (Step 22b) -- prioritize those
    # since that's this session's actual labeling work, then signboards, then
    # anything else still sitting on a placeholder.
    if "/books/" in path or "/newspaper/" in path:
        return 0
    if "/signboards/" in path:
        return 1
    return 2

all_rows = load_rows()
queue = sorted(
    [r["image"] for r in all_rows if r["text"] == "FILL_IN_URDU_TEXT_HERE"],
    key=folder_priority,
)
print(f"{len(queue)} images left to label "
      f"({sum(1 for p in queue if '/books/' in p)} books, "
      f"{sum(1 for p in queue if '/newspaper/' in p)} newspaper, "
      f"{sum(1 for p in queue if '/signboards/' in p)} signboards). "
      f"FLAGGED_ prefixed images are worth a skeptical look -- see Step 22b.")

# Load Step 22b's suggestions, if they exist, to pre-fill the text box below.
try:
    with open("data/line_suggestions.json", "r", encoding="utf-8") as f:
        line_suggestions = _json.load(f)
except FileNotFoundError:
    line_suggestions = {}

idx_box = {"i": 0}

img_widget = widgets.Image(format="png", layout=widgets.Layout(max_width="600px"))
progress_label = widgets.Label()
text_box = widgets.Textarea(
    placeholder="Type the Urdu text you see in the image...",
    layout=widgets.Layout(width="600px", height="80px"),
)
text_box.add_class("urdu-rtl")
display(widgets.HTML("<style>.urdu-rtl textarea{direction:rtl;text-align:right;font-size:18px;}</style>"))

save_btn = widgets.Button(description="Save & Next", button_style="success")
skip_btn = widgets.Button(description="Skip", button_style="warning")
back_btn = widgets.Button(description="\u2b05 Back")
out = widgets.Output()

def show_current():
    with out:
        clear_output(wait=True)
        i = idx_box["i"]
        if i >= len(queue):
            print("\U0001F389 All done! No more unlabeled images.")
            img_widget.value = b""
            text_box.value = ""
            progress_label.value = ""
            return
        path = queue[i]
        suggestion = line_suggestions.get(path)
        conf_note = (f"  (suggestion confidence: {suggestion['confidence']:.2f} -- verify before saving)"
                     if suggestion and suggestion.get("text") else "")
        progress_label.value = f"Image {i+1} / {len(queue)}  \u2014  {path}{conf_note}"
        with open(path, "rb") as f:
            img_widget.value = f.read()
        text_box.value = suggestion["text"] if suggestion else ""
        display(progress_label, img_widget)

def on_save(b):
    i = idx_box["i"]
    if i >= len(queue):
        return
    text = text_box.value.strip()
    if not text:
        with out:
            print("\u26a0\ufe0f Type something before saving (or hit Skip).")
        return
    save_one_label(queue[i], text)
    idx_box["i"] += 1
    show_current()

def on_skip(b):
    idx_box["i"] += 1
    show_current()

def on_back(b):
    if idx_box["i"] > 0:
        idx_box["i"] -= 1
    show_current()

save_btn.on_click(on_save)
skip_btn.on_click(on_skip)
back_btn.on_click(on_back)

display(widgets.HBox([back_btn, skip_btn, save_btn]), text_box, out)
show_current()
